In [2]:
!pip install -q "transformers<5" av opencv-python-headless
import av
print("PyAV installed successfully!")
print(av.__version__)

PyAV installed successfully!
19.0.2


In [4]:
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(video_path)

Saving 19233861-uhd_3840_2160_30fps.mp4 to 19233861-uhd_3840_2160_30fps.mp4
19233861-uhd_3840_2160_30fps.mp4


In [5]:
from transformers import pipeline
import gc

video_clf = pipeline(
    "video-classification",
    model="MCG-NJU/videomae-base-finetuned-kinetics"
)

result = video_clf(video_path, top_k=3)

print("Video Classification Results:")
print(result)

del video_clf
gc.collect()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu


Video Classification Results:
[{'score': 0.1499166339635849, 'label': 'recording music'}, {'score': 0.0735093504190445, 'label': 'archery'}, {'score': 0.03041601926088333, 'label': 'using remote controller (not gaming)'}]


196

In [8]:
from transformers import pipeline
from PIL import Image
import gc

image_clf = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

for i, frame in enumerate(frames):
    pil_image = Image.fromarray(frame)
    result = image_clf(pil_image, top_k=3)

    print(f"\nFrame {i+1} Results:")
    print(result)

del image_clf
gc.collect()

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cpu



Frame 1 Results:
[{'label': 'tripod', 'score': 0.40771764516830444}, {'label': 'reflex camera', 'score': 0.13072334229946136}, {'label': 'television, television system', 'score': 0.060044173151254654}]

Frame 2 Results:
[{'label': 'tripod', 'score': 0.2976337671279907}, {'label': 'television, television system', 'score': 0.18580780923366547}, {'label': 'reflex camera', 'score': 0.09529458731412888}]

Frame 3 Results:
[{'label': 'tripod', 'score': 0.3338112235069275}, {'label': 'television, television system', 'score': 0.12254543602466583}, {'label': 'reflex camera', 'score': 0.09328212589025497}]

Frame 4 Results:
[{'label': 'tripod', 'score': 0.36899518966674805}, {'label': 'television, television system', 'score': 0.13329453766345978}, {'label': 'reflex camera', 'score': 0.07617412507534027}]


0

In [10]:
import subprocess
import os
from transformers import pipeline
import gc

audio_path = "audio.wav"

result = subprocess.run(
    ["ffmpeg", "-y", "-i", video_path, "-vn",
     "-ar", "16000", "-ac", "1", audio_path],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("No audio track found in this video.")
    print("Audio transcription skipped.")
else:
    print("Audio extracted successfully!")

    asr = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-tiny"
    )

    transcription = asr(audio_path, return_timestamps=True)
    print("Audio Transcription:")
    print(transcription)

    del asr
    gc.collect()

No audio track found in this video.
Audio transcription skipped.
